In [21]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import adi
import time
import binascii

In [22]:
def text_to_bin(text):
    temp_bin_str = ''.join(format(ord(c), '07b') for c in text)
    bin_data = np.array([int(bit) for bit in temp_bin_str], dtype=int)
    return bin_data

def bin_to_text(binary_arr):
    bin_str = "".join(str(bit) for bit in binary_arr)
    text = ""
    for i in range(0, len(bin_str), 7):
        byte = bin_str[i:i+7]
        if len(byte) == 7 :
          text += chr(int(byte, 2))
    return text

def append_crc_to_frame(payload):
    byte_data = np.packbits(payload)  
    crc = binascii.crc_hqx(byte_data, 0xFFFF)  
    crc_bits = np.array(list(np.binary_repr(crc, width=16)), dtype=np.uint8)  
    return np.concatenate((payload, crc_bits))

def verify_crc(received_frame):
    received_payload = received_frame[:-16]
    received_crc = received_frame[-16:]
    byte_data = np.packbits(received_payload)  
    crc = binascii.crc_hqx(byte_data, 0xFFFF)  
    expected_crc = np.array(list(np.binary_repr(crc, width=16)), dtype=np.uint8)  
    if np.array_equal(received_crc, expected_crc):
        return True  
    else:
        return False

def int_to_fixed_binary_array(n, bits=8):
    return [int(bit) for bit in format(n, f'0{bits}b')]

def binary_array_to_int(binary_array):
    return int("".join(map(str, binary_array)), 2)

In [23]:
sample_rate = 10e6 # Hz
carrier_freq = 980e6 # Hz
num_samps = 100000 

sdr = adi.Pluto("ip:192.168.2.1")
sdr.sample_rate = int(sample_rate)
sdr.tx_rf_bandwidth = int(sample_rate) 
sdr.tx_lo = int(carrier_freq)
sdr.tx_hardwaregain_chan0 = -15
sdr.tx_cyclic_buffer = False 

sdr.rx_lo = int(carrier_freq)
sdr.rx_rf_bandwidth = int(sample_rate)
sdr.rx_buffer_size = num_samps
sdr.gain_control_mode_chan0 = 'slow_attack'

frame_id = 1

In [24]:
text_message = '''Mental health is vital to overall well-being, influencing how we think, feel, and act. 
It shapes relationships, decisions, and coping abilities. Yet, stigma often prevents people from seeking help. 
Common conditions like depression, anxiety, and stress are treatable with support. 
Promoting mental health means creating safe spaces, encouraging openness, and providing care access. 
Early recognition of distress—such as sadness, withdrawal, or sleep changes—enables timely help. 
Therapy, medication, and self-care practices like mindfulness and exercise improve resilience. 
Schools, workplaces, and governments must prioritize awareness, reduce stigma, and invest in accessible services. Mental health is essential, not optional.
'''

In [25]:
for mm in range(50):    
    t1 = time.time()
    print('frame id', frame_id)
    sdr.tx_enabled = True
    text = text_message[(frame_id-1)*40:(frame_id)*40]
    message_bits = text_to_bin(text)
    frame_no = int_to_fixed_binary_array(frame_id, 10)
    payload = np.concatenate((frame_no, message_bits))
    data_bits = append_crc_to_frame(payload)
    bpsk_symbols = np.array([-1 if bit == 0 else 1 for bit in data_bits])
    barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
    barker_appended_bpsk = np.concatenate((barker_code,  bpsk_symbols))
    sps = 8
    zero_padded = np.zeros(len(barker_appended_bpsk) * sps)
    zero_padded[::sps] = barker_appended_bpsk
    num_taps = 101
    beta = 0.34
    Ts = sps
    t = np.arange(num_taps) - (num_taps-1)//2
    rc_pulse = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)
    ps_conv_output = np.convolve(zero_padded, rc_pulse, mode = 'full')
    pulse_shaped = ps_conv_output[num_taps//2: -1-num_taps//2]
    tx_signal = pulse_shaped * (2**14)
    ####################################################################
    print('transmitting start')
    t_start = time.time()
    t_end = 0
    while(t_end < 3):
        t_end = time.time() - t_start
        sdr.tx(tx_signal)
    print('transmitting end')
    print(time.time() - t1)
    ######################################################################
    sdr.tx_destroy_buffer()
    sdr.tx_enabled = False
    sdr.rx_enabled = True
    time.sleep(0.05)
    flag = 0
    print('receving ack start')
    for j in range(4):
        for _ in range(3):
            _ = sdr.rx() 
        rx_samples = sdr.rx()
        energy = np.sum(np.abs(rx_samples)**2)
        avg_power = energy / len(rx_samples)
        sps = 8
        num_taps = 101
        beta = 0.34
        Ts = sps 
        t = np.arange(num_taps) - (num_taps-1)//2
        rc_pulse = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)
        matched_filter =  rc_pulse
        MF_op = np.convolve(rx_samples, matched_filter, mode='same')
        max_allowed = 1.5
        rx_max = np.max(np.abs(MF_op))
        scale_coarse = max_allowed / rx_max
        normalized_rx = MF_op * scale_coarse
        squared_FO = normalized_rx**2
        fs = sample_rate
        psd = np.fft.fftshift(np.abs(np.fft.fft(squared_FO)))
        f = np.linspace(-fs/2.0, fs/2.0, len(psd))
        max_freq = f[np.argmax(psd)]
        Ts = 1/fs
        t = np.arange(0, Ts*len(normalized_rx), Ts) # create time vector
        coarse_corrected = normalized_rx * np.exp(-1j*2*np.pi*max_freq*t/2.0)
        power = np.mean(np.abs(coarse_corrected)**2)
        normalized_2 = coarse_corrected / np.sqrt(power)
        samples = normalized_2
        N = len(samples)
        phase = 0
        freq = 0
        alpha = 0.132
        beta = 0.00932
        fine_correct = np.zeros(N, dtype=np.complex64)
        freq_log = []
        for i in range(N):
            fine_correct[i] = samples[i] * np.exp(-1j*phase)
            error = np.real(fine_correct[i]) * np.imag(fine_correct[i]) 
            freq += (beta * error)
            freq_log.append(freq * fs / (2*np.pi)) 
            phase += freq + (alpha * error)
            while phase >= 2*np.pi:
                phase -= 2*np.pi
            while phase < 0:
                phase += 2*np.pi
        samples  = fine_correct[100:]
        samples_interpolated = signal.resample_poly(samples, 16, 1)
        mu = 0 
        out = np.zeros(len(samples) + 10, dtype=np.complex64)
        out_rail = np.zeros(len(samples) + 10, dtype=np.complex64) #
        i_in = 0 
        i_out = 2 
        while i_out < len(samples) and i_in+16 < len(samples):
            out[i_out] = samples_interpolated[i_in*16 + int(mu*16)]
            out_rail[i_out] = int(np.real(out[i_out]) > 0) + 1j*int(np.imag(out[i_out]) > 0)
            x = (out_rail[i_out] - out_rail[i_out-2]) * np.conj(out[i_out-1])
            y = (out[i_out] - out[i_out-2]) * np.conj(out_rail[i_out-1])
            mm_val = np.real(y - x)
            mu += sps + 0.3*mm_val
            i_in += int(np.floor(mu))
            mu = mu - np.floor(mu) 
            i_out += 1 
        out = out[2:i_out] 
        time_sync = out
        barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
        barker_correlated = np.correlate(time_sync.real, barker_code, mode='full')
        offset  = np.argmax(np.abs(barker_correlated))+1
        peak = np.real(barker_correlated)[offset-1]
        # print('peak', peak)
        if peak < 0:
            time_sync = -1*time_sync
        payload_size = 210
        n_bits = payload_size + 16 + 10
        fr_sync_op = time_sync[offset:offset+n_bits]
        recovered_bits = (fr_sync_op > 0).astype(int)
        received_frame = recovered_bits
        if verify_crc(received_frame):
            decoded_text = bin_to_text(recovered_bits[10:payload_size+10])
            # print('Decoded text:', decoded_text)
            ack_id = binary_array_to_int(received_frame[:10])
            # print('ack id', ack_id)
            
            if ack_id == frame_id:
                flag = 1
    print('receving ack end')
    print(time.time() - t1)
    if flag == 1:
        frame_id = frame_id + 1
    sdr.rx_enabled = False
    time.sleep(0.5)


frame id 1
transmitting start
transmitting end
3.002427101135254
receving ack start
receving ack end
5.267584562301636
frame id 1
transmitting start
transmitting end
3.002643585205078
receving ack start
receving ack end
5.313641548156738
frame id 2
transmitting start
transmitting end
3.0039546489715576
receving ack start
receving ack end
5.256652593612671
frame id 3
transmitting start
transmitting end
3.00376296043396
receving ack start
receving ack end
5.216361999511719
frame id 3
transmitting start
transmitting end
3.002887487411499
receving ack start
receving ack end
5.222176551818848
frame id 4
transmitting start
transmitting end
3.003030776977539
receving ack start
receving ack end
5.268782138824463
frame id 5
transmitting start
transmitting end
3.0028011798858643
receving ack start
receving ack end
5.245391368865967
frame id 6
transmitting start
transmitting end
3.0035383701324463
receving ack start
receving ack end
5.2165446281433105
frame id 6
transmitting start
transmitting en

KeyboardInterrupt: 

In [ ]:
# samples = sdr.rx()
# plt.figure(figsize=(4, 4)) 
# plt.scatter(np.real(samples), np.imag(samples), s=1, alpha=0.5) 
# plt.title("Scatter Plot of Real vs Imaginary Parts of Samples")
# plt.xlabel("Real Part")
# plt.ylabel("Imaginary Part")
# plt.grid(True)
# plt.axis('equal') 
# plt.show()

In [ ]:
a = 'ACKACKACKACKACKACKACKACKACKACK'
len(a)

30